In [ ]:
# ==========================================
# OPTIONAL: Install required libraries
# ==========================================
# !pip install -q -r requirements.txt

In [ ]:
!pip install catboost

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#default imports
import pandas as pd
import numpy as np
import joblib
import os
import re
import json
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce

from sklearn.experimental import enable_iterative_imputer
from sklearn.preprocessing import LabelEncoder

#models
from sklearn.ensemble import ExtraTreesClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

#import custom pipelines
import sys
main_path = '/content/drive/MyDrive/FHI_Prediction/'
sys.path.append(main_path)
from Custom_Transformers.custom_engineering import FHIBaseSignals,FHIAdvancedSignals,DataCleaner,FeatureNameSanitizer,custom_smote_ratios,DropColumns
from Custom_Transformers.custom_patterns import UnsupervisedPatternExtractor
from Custom_Transformers.custom_preprocessing import FHIDataCleaner

import warnings
warnings.filterwarnings('ignore')


In [ ]:
import random
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [ ]:
exp_code_used = '1773924086362'
final_file_path = main_path  + f'FHI_Models/' + exp_code_used + '/'
raw_models_path = final_file_path + f'raw_models/'
cleaned_models_path = final_file_path + f'cleaned_models/'

In [ ]:
# Setup Data
le = LabelEncoder()

# Setup Data
data_path = '/content/drive/MyDrive/FHI_Prediction/Data/'
test_data = pd.read_csv(data_path + "Test.csv")
train_data = pd.read_csv(data_path + "Train.csv")
model_results_raw = pd.read_csv(final_file_path + "models_results_raw.csv")
model_results_clean = pd.read_csv(final_file_path + "models_results_clean.csv")

#encode the targets
y_enc = le.fit_transform(train_data['Target']) # [0, 1, 2] #{High:0, Low:1, Medium:2}
print(np.unique(y_enc, return_counts=True))

X_test_clean = test_data.drop(columns=['ID'] , errors='ignore').copy()
for col in X_test_clean.select_dtypes(include=['float64']).columns:
        X_test_clean[col] = X_test_clean[col].astype('float32')

In [ ]:
display(model_results_raw)

In [ ]:
display(model_results_clean)

In [ ]:
def load_trained_models(folder_name):
    """
    Loads all .joblib models from a folder into a dictionary
    """
    trained_models = {}

    if not os.path.exists(folder_name):
        print(f"Folder '{folder_name}' not found.")
        return trained_models

    for filename in os.listdir(folder_name):
        if filename.endswith(".joblib"):
            name = filename.replace(".joblib", "")
            filepath = os.path.join(folder_name, filename)
            trained_models[name] = joblib.load(filepath)
            print(f" Loaded: {name}")

    return trained_models

In [ ]:
print("loading raw models")
raw_models = load_trained_models(raw_models_path)

print("loading cleaned models")
cleaned_models = load_trained_models(cleaned_models_path)

In [ ]:
def load_ensemble_weights(filename):
    """Loads the dictionary of Optuna weights from a JSON file."""
    if not os.path.exists(filename):
        print(f"{filename} not found. Returning None.")
        return None

    with open(filename, 'r') as f:
        weights_dict = json.load(f)
    print(f"Loaded weights from {filename}:\n{json.dumps(weights_dict, indent=2)}")
    return weights_dict

In [ ]:
raw_saved_weights = load_ensemble_weights(final_file_path +  f'raw_ensemble_weights.json')
cleaned_saved_weights = load_ensemble_weights(final_file_path + f'cleaned_ensemble_weights.json')

In [ ]:
def baseline_submission(X_test_clean,le,raw_models):
    """
    This function generates the blended submission based on only the models trained on the raw dataset.
    """

    # Initialize the empty probability matrix
    n_samples = len(X_test_clean)
    n_classes = len(le.classes_) # This dynamically grabs your 3 classes (Low, Medium, High)
    blended_test_probs = np.zeros((n_samples, n_classes))

    raw_ensemble_weights = load_ensemble_weights(final_file_path + f'raw_ensemble_weights.json')

    print("Fusion Engine Started...")
    for name, model in raw_models.items():
        # The Gatekeeper: Only permit models that Optuna assigned >1% weight to contribute
        if name in raw_ensemble_weights and raw_ensemble_weights[name] > 0.01:
            print(f" Extracting {name} (Weight: {raw_ensemble_weights[name]:.3f})")

            # Predict test probabilities and scale them by their assigned weight
            test_probs = model.predict_proba(X_test_clean)
            blended_test_probs += test_probs * raw_ensemble_weights[name]

    # Resolve the final class by picking the highest weighted probability (Argmax)
    final_preds = np.argmax(blended_test_probs, axis=1)

    # Translate integer predictions (0, 1, 2) back into strings ("High", "Low", "Medium")
    final_labels = le.inverse_transform(final_preds)

    # Create the final submission file
    submission = pd.DataFrame({
        "ID": test_data["ID"],
        "Target": final_labels
    })

    # Save the final file dynamically appended with the local validation score
    sub_name = main_path + f'submission_Ensemble_Weighted_{exp_code_used}_raw.csv'
    submission.to_csv(sub_name, index=False)
    print(f"Saved to: {sub_name}")

    return submission

In [ ]:
base_submission =  baseline_submission(X_test_clean,le,raw_models)

In [ ]:
base_submission['Target'].value_counts()

##  Blend raw models with the models generated after cleaning the labels

In [ ]:
def blend_submissions(X_test_clean,y_enc,blend_weight):
    """
    This function generates the blended  submission based on models trained on the raw dataset
    and models trained on the cleaned dataset.
    """

    n_samples = len(X_test_clean)
    n_classes = len(np.unique(y_enc))

    raw_probs = np.zeros((n_samples, n_classes))
    clean_probs = np.zeros((n_samples, n_classes))

    raw_optuna_weights = load_ensemble_weights(final_file_path + 'raw_ensemble_weights.json')
    clean_optuna_weights = load_ensemble_weights(final_file_path + 'cleaned_ensemble_weights.json')

    # 2. Extract Raw Probabilities using Optuna Weights
    print(" Gathering Raw probabilities from the raw models")
    for name, model in raw_models.items():
        if name in raw_optuna_weights and raw_optuna_weights[name] > 0.01:
            weight = raw_optuna_weights[name]
            raw_probs += model.predict_proba(X_test_clean) * weight

    # 3. Extract Clean Probabilities using Optuna Weights
    print(" Gathering Clean probabilities from the cleaned models")
    for name, model in cleaned_models.items():
        if name in clean_optuna_weights and clean_optuna_weights[name] > 0.01:
            weight = clean_optuna_weights[name]
            clean_probs += model.predict_proba(X_test_clean) * weight

    # Resolve the final class by picking the highest weighted probability (Argmax)
    final_clean_preds = np.argmax(clean_probs, axis=1)

    # Translate integer predictions (0, 1, 2) back into strings ("High", "Low", "Medium")
    final_clean_labels = le.inverse_transform(final_clean_preds)

    # Create the final submission file
    cleaned_submission = pd.DataFrame({
        "ID": test_data["ID"],
        "Target": final_clean_labels
    })
    #save cleaned file
    cleaned_csv_name = main_path + f"submission_Ensemble_Weighted_{exp_code_used}_clean.csv"
    cleaned_submission.to_csv(cleaned_csv_name, index=False)
    print(f"Saved cleaned submission to {cleaned_csv_name}")

    # 4: Execute Probability Fusion. Blend the probabilities from the cleaned models and from the raw models
    blend_weight_clean = 1.0 - blend_weight
    print(f"Executing the {int(blend_weight*100)}/{int(blend_weight_clean*100)} Probability Fusion...")

    # Mathematical fusion of the probability matrices
    final_blended_probs = (raw_probs * blend_weight) + (clean_probs * blend_weight_clean)

    # Resolve probabilities to discrete class integers, then decode to strings
    final_preds = np.argmax(final_blended_probs, axis=1)
    final_labels = le.inverse_transform(final_preds)

    # Generate final submission
    hedge_submission = pd.DataFrame({
        "ID": test_data["ID"],
        "Target": final_labels
    })

    csv_name = main_path + f"submission_Pruned_Blend_{int(blend_weight*100)}_{int(blend_weight_clean*100)}_{exp_code_used}.csv"
    hedge_submission.to_csv(csv_name, index=False)
    print(f"Saved hedged submission to {csv_name}")
    return cleaned_submission,hedge_submission

In [ ]:
cleaned_submission,hedge_submission= blend_submissions(X_test_clean,y_enc,blend_weight=0.5,)

In [ ]:
cleaned_submission['Target'].value_counts()

In [ ]:
hedge_submission['Target'].value_counts()

# FEATURE IMPORTANCE

In [ ]:
def plot_global_ensemble_importance(trained_models, ensemble_weights, top_n=10):
    """
    Calculates the true global feature importance by normalizing each model's
    internal importances and multiplying them by their Optuna ensemble weight.
    """
    importance_dfs = []

    print("Aggregating Weighted Feature Importances...")
    for name, model in trained_models.items():
        # Skip models that Optuna gave zero weight to
        if name not in ensemble_weights or ensemble_weights[name] <= 0.01:
            continue

        weight = ensemble_weights[name]

        try:
            # Extract classifier (the last step in pipeline)
            classifier_name, classifier = model.steps[-1]

            if not hasattr(classifier, 'feature_importances_'):
                continue

            raw_importances = classifier.feature_importances_

            # 1. NORMALIZE: Convert raw scores to percentages (0.0 to 1.0)
            norm_importances = raw_importances / np.sum(raw_importances)

            # 2. EXTRACT NAMES: Handle both Scikit-Learn and Custom Transformers
            try:
                feature_names = model[:-1].get_feature_names_out()
            except:
                if hasattr(classifier, 'feature_names_in_'):
                    feature_names = classifier.feature_names_in_
                else:
                    feature_names = [f"Feature_{i}" for i in range(len(norm_importances))]

            # 3. APPLY OPTUNA WEIGHT: Multiply normalized score by the model's voting power
            df = pd.DataFrame({
                'Feature': feature_names,
                name: norm_importances * weight
            })
            importance_dfs.append(df)
            print(f" Processed {name} (Weight: {weight:.3f})")

        except Exception as e:
            print(f" Could not extract importance for {name}: {e}")

    # 4. MERGE: Combine all dataframes on 'Feature', filling mismatches with 0
    final_df = reduce(lambda left, right: pd.merge(left, right, on='Feature', how='outer'), importance_dfs)
    final_df = final_df.fillna(0)

    # 5. CALCULATE GLOBAL SCORE: Sum across all the weighted model columns
    model_cols = [c for c in final_df.columns if c != 'Feature']
    final_df['Global_Weighted_Importance'] = final_df[model_cols].sum(axis=1)

    # Sort for plotting
    final_df = final_df.sort_values(by='Global_Weighted_Importance', ascending=False).head(top_n)

    # 6. plot
    plt.figure(figsize=(12, 8))
    sns.barplot(
        data=final_df,
        x='Global_Weighted_Importance',
        y='Feature',
        hue='Feature',
        palette='viridis',
        legend=False
    )
    plt.title("Global Ensemble Feature Importance\n(Normalized & Weighted by Optuna)", fontsize=16, fontweight='bold')
    plt.xlabel("Weighted Contribution Score", fontsize=12)
    plt.ylabel("Feature Name", fontsize=12)

    save_path = final_file_path +  f"Global_feat_importance.png"

    # dpi=300 makes it high-resolution (HD), bbox_inches='tight' stops names from getting cut off
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.tight_layout()
    plt.show()

    return final_df

In [ ]:
global_fi_df = plot_global_ensemble_importance(raw_models, raw_saved_weights)

# SHAP

In [ ]:
def plot_shap_direction(model_filepath, X_val, class_index=2, class_name="Low Health"):
    print(f"Generating SHAP Explainer for Class: {class_name}...")

    # 1. Load the pipeline
    pipeline = joblib.load(model_filepath)
    classifier = pipeline.steps[-1][1]

    # 2. Safely transform the data step-by-step
    Xt = X_val.copy()
    feature_names = list(X_val.columns)

    for name, step in pipeline.steps[:-1]:
        if hasattr(step, 'transform'):
            Xt = step.transform(Xt)
            # Catch feature names as they update
            if hasattr(step, 'get_feature_names_out'):
                try:
                    feature_names = step.get_feature_names_out()
                except Exception:
                    pass

    # 3. Force the data into a clean, dense numpy array
    if hasattr(Xt, 'toarray'):
        Xt = Xt.toarray()
    elif hasattr(Xt, 'todense'):
        Xt = Xt.todense()

    Xt = np.array(Xt, dtype=np.float32)

    # Try one last time to get feature names directly from the Random Forest
    if hasattr(classifier, 'feature_names_in_'):
        feature_names = classifier.feature_names_in_
    elif len(feature_names) != Xt.shape[1]:
        feature_names = [f"Feature_{i}" for i in range(Xt.shape[1])]

    # 4. Create the final DataFrame and KILL any NaNs that cause the gray lines
    X_shap_df = pd.DataFrame(Xt, columns=feature_names)
    X_shap_df = X_shap_df.fillna(0)

    # 5. Initialize SHAP
    print("Calculating SHAP values (this may take a moment)...")
    explainer = shap.TreeExplainer(classifier)
    shap_values = explainer.shap_values(X_shap_df)

    if isinstance(shap_values, list):
        target_shap_values = shap_values[class_index]
    else:
        if len(shap_values.shape) == 3:
            target_shap_values = shap_values[:, :, class_index]
        else:
            target_shap_values = shap_values

    # 6. Plot and Save
    plt.figure(figsize=(10, 8))
    plt.title(f"SHAP Feature Impact: Predicting '{class_name}'\n(Model: {model_filepath.split('/')[-1]})",
              fontsize=14, fontweight='bold', pad=20)

    shap.summary_plot(target_shap_values, X_shap_df,max_display=10, show=False)

    clean_name = class_name.replace(' ', '_')
    save_path = final_file_path + f"SHAP_Impact_{clean_name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Plot successfully saved to: {save_path}")

    plt.tight_layout()
    plt.show()

In [ ]:
# 1. Take a small, fast sample from the test data you already cleaned
X_shap_sample = X_test_clean.sample(n=500, random_state=SEED)

print(f"Sampled {len(X_shap_sample)} rows from X_test_clean for SHAP analysis.")

# 2. Run the SHAP Direction function on your best model
plot_shap_direction(
    model_filepath=raw_models_path + 'RandomForest_Patterns.joblib',
    X_val=X_shap_sample,  # Pass the test sample here
    class_index=1,        # Low
    class_name="Low"
)

In [ ]:
plot_shap_direction(
    model_filepath=raw_models_path + 'RandomForest_Patterns.joblib',
    X_val=X_shap_sample,
    class_index=2,
    class_name="Medium"
)

In [ ]:
plot_shap_direction(
    model_filepath=raw_models_path + 'RandomForest_Patterns.joblib',
    X_val=X_shap_sample,
    class_index=0,
    class_name="High"
)